# Differential Equations — Session 4
## Section 2.1: Solution Curves Without Solving the Equation

**Planned length:** 90 minutes  
**Notebook type:** Student interactive lecture

### Learning objectives

By the end of the session, students should be able to:

1. Interpret $y'=f(x,y)$ as a field of local slopes.
2. Sketch and read a direction field.
3. Identify autonomous equations and equilibrium solutions.
4. Construct a phase line from the sign of $f(y)$.
5. Classify equilibria as stable, unstable, or semistable.
6. Predict monotonicity and long-term behavior without finding an explicit solution.
7. Explain the translation property of autonomous equations.

> The mathematical sequence follows the publisher's Section 2.1, but the examples, figures, and code are original.

**Edition:** Local Instructor Interactive Edition

> **Student interactive edition — local Jupyter/Cursor workflow**
>
> 1. Run the **Local notebook setup** cell below first.
> 2. Read each explanation and derivation in order.
> 3. Run simulation and visualization cells as you reach them.
> 4. At each **Classroom Checkpoint**, stop and work out your answer before class discussion continues.
> 5. This student edition intentionally contains **no instructor answer-reveal cells and no instructor solution notes**.
>
> **Tip:** During class, use `Shift + Enter` to move through the notebook one cell at a time.

In [ ]:
# Local notebook setup — run this cell first.
import importlib.util
import platform
import sys

_REQUIRED = ["numpy", "matplotlib", "scipy", "sympy", "ipywidgets"]
_missing = [name for name in _REQUIRED if importlib.util.find_spec(name) is None]

print(f"Python {sys.version.split()[0]} on {platform.system()}")
if _missing:
    print("Missing packages:", ", ".join(_missing))
    print("From the project folder, run:")
    print("python -m pip install -r requirements.txt")
else:
    print("Student notebook environment is ready.")

### Suggested pacing

| Time | Topic |
|---:|---|
| 0–15 min | Slopes and direction fields |
| 15–32 min | Reading qualitative behavior |
| 32–50 min | Autonomous equations and equilibria |
| 50–70 min | Phase lines and stability |
| 70–82 min | Numerical solution curves versus predictions |
| 82–88 min | Translation property |
| 88–90 min | Exit check |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import solve_ivp, quad
from scipy.special import erf
from IPython.display import display, Markdown

try:
    from ipywidgets import interact, FloatSlider, IntSlider, Dropdown
    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
np.set_printoptions(precision=5, suppress=True)

def slope_field(f, xlim=(-3, 3), ylim=(-3, 3), density=21, title=None):
    x = np.linspace(*xlim, density)
    y = np.linspace(*ylim, density)
    X, Y = np.meshgrid(x, y)
    S = np.asarray(f(X, Y), dtype=float)
    S = np.nan_to_num(S, nan=0.0, posinf=20.0, neginf=-20.0)
    U = np.ones_like(S)
    length = np.sqrt(U**2 + S**2)
    plt.quiver(X, Y, U/length, S/length, angles="xy", pivot="mid")
    plt.xlim(*xlim)
    plt.ylim(*ylim)
    plt.xlabel("x")
    plt.ylabel("y")
    if title:
        plt.title(title)

def euler_method(f, x0, y0, h, n_steps):
    xs = np.empty(n_steps + 1)
    ys = np.empty(n_steps + 1)
    xs[0], ys[0] = x0, y0
    for n in range(n_steps):
        ys[n+1] = ys[n] + h*f(xs[n], ys[n])
        xs[n+1] = xs[n] + h
    return xs, ys

print("Notebook ready.")
print("Interactive widgets available:", WIDGETS_AVAILABLE)

## Formal theory reference

### Definition 2.1-A — Direction field

For an equation

$$
y'=f(x,y),
$$

a **direction field** is a collection of short line segments drawn at selected points $(x,y)$, each with slope $f(x,y)$. A differentiable solution curve must be tangent to the field at every point through which it passes.

### Definition 2.1-B — Autonomous equation and equilibrium

A first-order equation is **autonomous** when it has the form

$$
y'=f(y).
$$

A number $c$ satisfying $f(c)=0$ is an **equilibrium point**, and the constant function $y(x)=c$ is an equilibrium solution.

### Theorem 2.1-C — Qualitative behavior between equilibria

Assume $f$ and $f'$ are continuous on an interval. Between any two consecutive equilibria of $y'=f(y)$:

1. $f(y)$ has one fixed sign.
2. Every nonconstant solution is strictly monotone.
3. A nonconstant solution cannot cross an equilibrium solution.
4. A bounded solution can approach only an equilibrium level as $x\to\infty$ or $x\to-\infty$.

### Stability criterion

For a simple equilibrium $c$:

- If $f'(c)<0$, then $c$ is asymptotically stable.
- If $f'(c)>0$, then $c$ is unstable.

When $f'(c)=0$, the derivative test is inconclusive and a phase-line sign analysis is required.

### Translation property

If $y(x)$ solves an autonomous equation, then $y(x-k)$ also solves it for every constant $k$.

### Classroom Checkpoint — Read the Phase Line

For

$$
y'=y(2-y),
$$

identify the equilibria and predict their stability without solving the equation.

> Pause here. Before continuing, try to explain their reasoning before continuing.

## 1. From one derivative to a field of slopes

For the first-order equation

$$
y'=f(x,y),
$$

the number $f(x,y)$ is the slope that a solution curve must have when it passes through $(x,y)$.

A **direction field** places a short line segment with slope $f(x,y)$ at many points. It lets us see possible solution behavior before solving the equation.

In [ ]:
# Direction field and several numerical solution curves
def f1(x, y):
    return x - y

slope_field(f1, xlim=(-3, 3), ylim=(-3, 3), density=23,
            title=r"Direction field for $y'=x-y$")

for y0 in [-2, -0.5, 1, 2.5]:
    sol = solve_ivp(lambda x, y: f1(x, y[0]), (-3, 3), [y0],
                    t_eval=np.linspace(-3, 3, 500))
    plt.plot(sol.t, sol.y[0], linewidth=2, label=fr"$y(-3)={y0}$")

plt.legend()
plt.show()

### Reading the field

For $y'=x-y$:

- Slopes are zero on the line $y=x$.
- Below $y=x$, $x-y>0$, so solutions increase.
- Above $y=x$, $x-y<0$, so solutions decrease.

The curve $y=x$ is a **zero-slope isocline**, but it is not itself a solution because substituting $y=x$ gives $y'=1$, not $0$.

## 2. Direction fields can reveal shape

Consider

$$
y'=0.3xy.
$$

The axes have zero slope. The signs of $x$ and $y$ determine whether a solution increases or decreases.

In [ ]:
def f2(x, y):
    return 0.3*x*y

slope_field(f2, xlim=(-4, 4), ylim=(-4, 4), density=23,
            title=r"Direction field for $y'=0.3xy$")

x_eval = np.linspace(-4, 4, 600)
for C in [-2, -0.8, 0.5, 1.5]:
    y_eval = C*np.exp(0.15*x_eval**2)
    plt.plot(x_eval, y_eval, linewidth=2, label=fr"$C={C}$")

plt.ylim(-4, 4)
plt.legend()
plt.show()

The exact family is

$$
y=Ce^{0.15x^2}.
$$

The field predicts the horseshoe-like behavior of positive solutions and the inverted behavior of negative solutions.

## 3. Autonomous equations

An equation is **autonomous** when the independent variable does not appear explicitly:

$$
y'=f(y).
$$

Every zero $c$ of $f$ produces an equilibrium solution

$$
y(x)=c.
$$

For autonomous equations, every point on the same horizontal line has the same slope.

In [ ]:
def autonomous_field(x, y):
    return y*(y+1)*(3-y)

slope_field(autonomous_field, xlim=(-3, 3), ylim=(-2.5, 4), density=25,
            title=r"Autonomous field for $y'=y(y+1)(3-y)$")

for level in [-1, 0, 3]:
    plt.axhline(level, linestyle="--", label=fr"equilibrium $y={level}$")

plt.legend()
plt.show()

## 4. Phase-line analysis

For

$$
y'=f(y)=y(y+1)(3-y),
$$

the critical points are

$$
y=-1,\qquad y=0,\qquad y=3.
$$

Check the sign of $f(y)$ on each interval:

| Interval | Sign of $f(y)$ | Motion |
|---|---:|---|
| $y<-1$ | $+$ | upward |
| $-1<y<0$ | $-$ | downward |
| $0<y<3$ | $+$ | upward |
| $y>3$ | $-$ | downward |

Therefore:

- $y=-1$ is asymptotically stable.
- $y=0$ is unstable.
- $y=3$ is asymptotically stable.

In [ ]:
# Original phase-line visualization
fig, ax = plt.subplots(figsize=(3.5, 7))
ax.set_xlim(-1, 1)
ax.set_ylim(-2.5, 4)
ax.axvline(0, linewidth=2)

equilibria = [-1, 0, 3]
for c in equilibria:
    ax.scatter([0], [c], s=90)

# arrows in each interval
for y_start, y_end in [(-2.2, -1.35), (-0.25, -0.75), (0.4, 1.2), (3.8, 3.25)]:
    ax.annotate("", xy=(0, y_end), xytext=(0, y_start),
                arrowprops=dict(arrowstyle="->", linewidth=2))

ax.text(0.18, -1, "stable", va="center")
ax.text(0.18, 0, "unstable", va="center")
ax.text(0.18, 3, "stable", va="center")
ax.set_xticks([])
ax.set_ylabel("y")
ax.set_title(r"Phase line for $y'=y(y+1)(3-y)$")
plt.show()

## 5. Test the qualitative prediction numerically

A uniqueness theorem implies that nonconstant solutions cannot cross equilibrium solutions. Each initial value remains in its original phase-line interval.

In [ ]:
def cubic_rhs(t, y):
    return y[0]*(y[0]+1)*(3-y[0])

t_eval = np.linspace(0, 8, 800)
for y0 in [-2, -0.6, 0.2, 1.5, 2.8, 3.6]:
    sol = solve_ivp(cubic_rhs, (0, 8), [y0], t_eval=t_eval,
                    rtol=1e-8, atol=1e-10)
    plt.plot(sol.t, sol.y[0], label=fr"$y(0)={y0}$")

for level in [-1, 0, 3]:
    plt.axhline(level, linestyle="--")
plt.xlabel("t")
plt.ylabel("y(t)")
plt.title("Solution curves confirm the phase-line predictions")
plt.legend(ncol=2)
plt.show()

## 6. Semistability

For

$$
y'=(y-1)^2,
$$

the equilibrium $y=1$ attracts solutions from below but repels solutions from above. It is **semistable**.

In [ ]:
def semi_rhs(t, y):
    return (y[0]-1)**2

for y0 in [-1, 0, 0.7, 1.2, 1.6]:
    sol = solve_ivp(semi_rhs, (0, 6), [y0],
                    t_eval=np.linspace(0, 6, 600),
                    rtol=1e-8, atol=1e-10)
    plt.plot(sol.t, sol.y[0], label=fr"$y(0)={y0}$")

plt.axhline(1, linestyle="--", label="semistable equilibrium")
plt.ylim(-1.5, 6)
plt.xlabel("t")
plt.ylabel("y(t)")
plt.title(r"One-sided attraction for $y'=(y-1)^2$")
plt.legend()
plt.show()

## 7. Translation property

If $y(x)$ solves an autonomous equation $y'=f(y)$, then

$$
y_k(x)=y(x-k)
$$

also solves the same equation.

For logistic growth,

$$
y'=y(1-y),
$$

changing the initial time only shifts the same S-shaped trajectory horizontally.

In [ ]:
x = np.linspace(-8, 8, 700)

for k in [-3, 0, 2.5]:
    y = 1/(1 + np.exp(-(x-k)))
    plt.plot(x, y, linewidth=2, label=fr"$y(x-{k})$")

plt.axhline(0, linestyle="--")
plt.axhline(1, linestyle="--")
plt.xlabel("x")
plt.ylabel("y")
plt.title("Autonomous solution curves are horizontal translations")
plt.legend()
plt.show()

## Interactive exploration — Initial value and basin of attraction

Move the initial value. The phase line predicts whether the solution approaches $-1$ or $3$, remains at $0$, or moves away outside the stable basins.

In [ ]:
def explore_autonomous(y0=1.0, final_time=8.0):
    def rhs(t, y):
        return [y[0]*(y[0]+1)*(3-y[0])]

    sol = solve_ivp(
        rhs, (0, final_time), [y0],
        t_eval=np.linspace(0, final_time, 700),
        rtol=1e-8, atol=1e-10
    )

    plt.plot(sol.t, sol.y[0], linewidth=2, label=fr"$y(0)={y0:.2f}$")
    for level in [-1, 0, 3]:
        plt.axhline(level, linestyle="--")
    plt.xlabel("t")
    plt.ylabel("y(t)")
    plt.title("Initial conditions select different long-term states")
    plt.legend()
    plt.show()

    print("Final displayed value:", sol.y[0, -1])

if WIDGETS_AVAILABLE:
    interact(
        explore_autonomous,
        y0=FloatSlider(min=-2.0, max=4.0, step=0.1, value=1.0),
        final_time=FloatSlider(min=2.0, max=15.0, step=1.0, value=8.0)
    )
else:
    explore_autonomous()

## Classroom Checkpoint — Exit Check

Without solving,

$$
y'=(y+2)(y-1)^2(y-4),
$$

identify the equilibria and classify their stability.

> Pause here. Let students commit to an answer before running the next cell.